In [3]:
!pip install langchain-openai

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.4 MB/s eta 0:00:00


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260314_langChain_practice2.ipynb)

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

from google.colab import userdata
userdata.get('OPENAI_API_KEY')

#api_key = os.getenv('OPENAI_API_KEY')
api_key = api_key = os.getenv('OPENAI_API_KEY')

In [17]:
api_key

In [ ]:
# chain = prompt | llm | parser
# LCEL : LangChain Expression Language
# LCEL을 도와주는 Runnable

In [ ]:
# RunnableSequence
# RunnableParallel
# RunnablePassthrough

In [4]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    summary = ChatPromptTemplate.from_template('{text}를 한 줄로 요약해 주세요.'),
    keywords = ChatPromptTemplate.from_template('{text}를 한 줄로 요약해 주세요.')
)


In [5]:
result = parallel_chain.invoke({'text': 'Langchain은 LLM 기반 application 개발 프레임워크입니다.'})

In [6]:
result

{'summary': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.를 한 줄로 요약해 주세요.', additional_kwargs={}, response_metadata={})]),
 'keywords': ChatPromptValue(messages=[HumanMessage(content='Langchain은 LLM 기반 application 개발 프레임워크입니다.를 한 줄로 요약해 주세요.', additional_kwargs={}, response_metadata={})])}

In [20]:
import os
from dotenv import load_dotenv
load_dotenv()

from google.colab import userdata

# Retrieve API key from Colab secrets
api_key = userdata.get('OPENAI_API_KEY')

# Initialize ChatOpenAI with the retrieved API key
llm = ChatOpenAI(model='gpt-4o-mini', openai_api_key=api_key)

In [28]:
# RunnableBranch
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import StrOutputParser

tech_chain = ChatPromptTemplate.from_template(' 기술지원팀입니다 : {question}') | llm | StrOutputParser()
billing_chain = ChatPromptTemplate.from_template('요금 관리 팀입니다 : {question}') | llm | StrOutputParser()
general_chain = ChatPromptTemplate.from_template(' 일반 상담 팀입니다 : {question}') | llm | StrOutputParser()

# 각 파이프라인 분기를 구분해 주는 Branch 함수
def route_logic(x):
  text = x['question']
  if '오류' in text or '에러' in text:
    return 'technical'
  elif '가격' in text or '요금' in text:
    return 'billing'
  return 'general'

branch = RunnableBranch(
    (lambda x : route_logic(x) == 'technical', tech_chain),
    (lambda x : route_logic(x) == 'billing', billing_chain),
    general_chain
)

questions = [
    '프린트 오류가 났어', '월 요금이 얼마야', '영업시간을 알려줘'
]
for q in questions:
  print(f"Q: {q}\nA: {branch.invoke({'question': q})}\n")

Q: 프린트 오류가 났어
A: 안녕하세요! 프린트 오류에 대해 도움을 드리겠습니다. 다음의 몇 가지 단계를 시도해 보세요:

1. **프린터 전원 확인**: 프린터가 켜져 있는지 확인하고, 전원이 정상적으로 공급되고 있는지 체크하세요.

2. **연결 상태 확인**: 프린터와 컴퓨터 간의 연결(USB 케이블 또는 Wi-Fi)이 제대로 되어 있는지 점검하세요.

3. **종이 걸림 확인**: 프린터 내부에 종이가 걸려 있는지 확인하고, 걸린 종이를 제거하세요.

4. **인쇄 대기열 확인**: 컴퓨터의 인쇄 대기열에서 인쇄 작업이 멈춰있거나 오류가 발생한 작업이 있는지 확인하고, 필요하다면 삭제 후 다시 인쇄해 보세요.

5. **드라이버 업데이트**: 프린터 드라이버가 최신 버전인지 확인하고, 필요하다면 업데이트하세요.

6. **프린터 상태 점검**: 프린터가 정상 동작하는지 테스트 페이지를 인쇄해 보세요.

이 방법으로 문제가 해결되지 않으면, 자세한 오류 메시지나 상황을 말씀해 주시면 추가로 도와드리겠습니다.

Q: 월 요금이 얼마야
A: 월 요금은 서비스나 제품에 따라 다를 수 있습니다. 어떤 서비스나 제품의 월 요금을 알고 싶으신지 구체적으로 말씀해 주시면 더 정확한 정보를 제공해 드릴 수 있습니다. 예를 들어, 통신 요금, 구독 서비스 요금 등 여러 종류가 있습니다. 추가 정보를 주시면 감사하겠습니다!

Q: 영업시간을 알려줘
A: 안녕하세요! 일반 상담 팀입니다. 영업시간에 대한 정보는 각 회사나 기관에 따라 다를 수 있습니다. 특정한 회사나 기관의 영업시간을 알고 싶으신 건가요? 혹은 일반적인 영업시간에 대해 궁금하신 건가요? 더 구체적인 정보를 주시면 도움을 드리겠습니다.



In [30]:
# 배포를 쉽고 도와주는 도구
!pip install gradio

In [31]:
import gradio as gr

In [34]:
def greet(name):
  return f'안녕하세요! {name}님'

demo1 = gr.Interface(
    fn = greet, # 함수
    inputs = gr.Textbox(label='이름 입력'), # 입력값 받기
    outputs = gr.Textbox(label='인사말'),
    title = '인사봇'
)

demo1.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74f3ee0056a007d6c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
demo1.close()

Closing server running on port: 7860


In [37]:
def echo_bot(message, history):
  return f"Echo: {message}"

demo2 = gr.ChatInterface(
    fn = echo_bot,
    title = '에코봇',
    examples = ['안녕하세요', '오늘 날씨 어때요?', 'FAQ 챗봇 테스트']
)

demo2.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8eb7562b335f0fe3c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [39]:
demo2.close()

Closing server running on port: 7861


In [43]:
with gr.Blocks(title='custom layout') as demo3:
  gr.Markdown('custom layout demo')
  with gr.Row():
    with gr.Column(scale=2):
      input_text = gr.Textbox(label='질문')
      submit_btn = gr.Button('전송')
    with gr.Column(scale=1):
      category_output = gr.Textbox(label='카테고리')

  output_text = gr.Textbox(label='답변')

  def process(text):
    cat = '기술' if any(kw in text for kw in ['오류', '설치', '연결']) else '일반'
    return cat, f'[{cat}] {text}'

  submit_btn.click(fn=process, inputs=input_text, outputs=[category_output, output_text])

demo3.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc95e6bc39dea3dda9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [44]:
demo3.close()

Closing server running on port: 7861


In [45]:
type(llm)

langchain_openai.chat_models.base.ChatOpenAI

In [49]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# llm 붙이기
def echo_bot(message, history):
  return f"Echo: {message}"

def chat_with_llm(message, history):
  messages = [SystemMessage(content = '당신은 친절한 한국어 어시스턴트입니다.')]
  for human, ai in history:
    messages.append(HumanMessage(content=human))
    messages.append(AIMessage(content=ai))

  # 그라디오 버전이 달라서 history가 자동으로 작동되지 않는 경우에 아래처럼 변환.. (튜플, 딕셔너리 차이)
  # for mmm in history:
  #   if mmm['role'] == 'user':
  #     messages.append(HumanMessage(content=mmm['content']))
  #   else:
  #     messages.append(AIMessage(content=mmm['content']))

  messages.append(HumanMessage(content=message))
  return llm.invoke(messages).content

demo4 = gr.ChatInterface(
    fn = chat_with_llm,
    title = 'Chatbot',
    examples = ['안녕하세요', 'LLM에 대해 알려주세요', 'FAQ 챗봇 테스트']
)

demo4.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e6b6f062aa8221fa4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
# llm의 지식을 주입
faq_data = [
    {"category": "계정", "question": "비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?",
     "answer": "IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다."},
    {"category": "계정", "question": "계정이 잠겼습니다. 어떻게 해제하나요?",
     "answer": "5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요."},
    {"category": "계정", "question": "신규 계정은 어떻게 만드나요?",
     "answer": "신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다."},
    {"category": "계정", "question": "2단계 인증(MFA)을 설정하려면?",
     "answer": "IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."},
]

In [53]:
faq_context = '\n'.join([f"Q: {item['question']}\nA: {item['answer']}" for item in faq_data])

In [54]:
faq_context

"Q: 비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?\nA: IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다.\nQ: 계정이 잠겼습니다. 어떻게 해제하나요?\nA: 5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요.\nQ: 신규 계정은 어떻게 만드나요?\nA: 신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다.\nQ: 2단계 인증(MFA)을 설정하려면?\nA: IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요."

In [56]:
prompt = ChatPromptTemplate.from_messages([
    ("system", f"너는 사내 IT 지원팀 챗봇이야.\n아래 제공된 FAQ 데이터를 바탕으로 사용자 질문에 친절하게 답해줘.\n데이터에 없는 내용은 'IT 헬프데스크(1234번)으로 문의해주세요' 라고 안내해줘\n\n[FAQ 데이터]\n{faq_context}"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

def chat_response(message, history):
    response = chain.invoke({"question": message})
    return response

demo = gr.ChatInterface(
    fn = chat_response,
    title = 'Chatbot',
    examples = ["비밀번호를 어떻게 초기화하나요?", "wi-fi가 느려요", "2단계 인증은 어떻게 설정하나요?"],
    description = '무엇을 도와드릴까요?'
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f72c78fbe0ee5462e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
